<h1 style="font-family: 'Times New Roman'; text-align: center;"><b>Machine Learning</b></h1>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b>URL Phishing Detection Project</b></h2>

<h3 style="font-family: 'Times New Roman'; text-align: right;"><b>- Pallav Biyala (24110234)</b></h3>

<hr>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b> Support Vector Machines </b></h2>

### Importing Libraries

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,balanced_accuracy_score, recall_score
from sklearn.exceptions import ConvergenceWarning

In [2]:
df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")

<hr>

### Preprocessing the Dataset:

In [3]:
# Remove repeated URLs before splitting.
df = df.drop_duplicates(subset=["URL"], keep="first").copy()

# All 50 numeric input features, excluding the target.
all_numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != "label"
]

print("Total numeric features:", len(all_numeric_features))

# M1: Strict lexical URL features.
lexical_features = [
    "URLLength", "DomainLength", "TLDLength", "NoOfSubDomain",
    "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL",
    "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL", "IsDomainIP",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio"
]

# M2: Add the three derived statistical features.
derived_features = [
    "TLDLegitimateProb",
    "URLCharProb",
    "CharContinuationRate"
]
# M3 adds HTTPS along with lexial and derived features --> represents what we can have if we are just given URL
# M4: Add HTTPS and the page-dependent features.
# URLSimilarityIndex is deliberately reserved for M5.
page_features = [
    c for c in all_numeric_features
    if c not in lexical_features
    and c not in derived_features
    and c not in ["IsHTTPS", "URLSimilarityIndex"]
]

feature_sets = {
    "M1 - Lexical": lexical_features,

    "M2 - Derived": (
        lexical_features + derived_features
    ),

    "M3 - URL Features": (
        lexical_features + derived_features + ["IsHTTPS"]
    ),

    "M4 - Page-aware": (
        lexical_features
        + derived_features
        + ["IsHTTPS"]
        + page_features
    ),

    "M5 - All numeric": all_numeric_features
}

for name, features in feature_sets.items():
    missing = set(features) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"
    assert len(features) == len(set(features)), (
        f"{name}: duplicate feature names"
    )
    print(f"{name}: {len(features)} features")

assert "URLSimilarityIndex" not in feature_sets["M4 - Page-aware"]
assert "URLSimilarityIndex" in feature_sets["M5 - All numeric"]

Total numeric features: 50
M1 - Lexical: 17 features
M2 - Derived: 20 features
M3 - URL Features: 21 features
M4 - Page-aware: 49 features
M5 - All numeric: 50 features


In [4]:
X = df[all_numeric_features]
y = df["label"]

# 80% train+validation, 10% test.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

# Split the remaining 90% into 80% train and 10% validation
# of the original dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=1/9,
    random_state=42,
    stratify=y_train_val
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 188296
Validation: 23537
Test: 23537


<hr>

### Training our Model

In [5]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C = 1.0,
        max_iter = 3000,
        random_state = 42
    ))
])

for name, features in feature_sets.items():
    svm_pipeline.fit(
        X_train[features],
        y_train
    )

    y_val_pred = svm_pipeline.predict(
        X_val[features]
    )
    
    print(name, ":\n")
    print("Accuracy: ", accuracy_score(y_val, y_val_pred))
    print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_val_pred, labels=[0, 1]))
    print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

M1 - Lexical :

Accuracy:  0.9955389386922717
Balanced Accuracy: 0.9947771587743732

Confusion Matrix:
 [[ 9947   105]
 [    0 13485]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99     10052
           1       0.99      1.00      1.00     13485

    accuracy                           1.00     23537
   macro avg       1.00      0.99      1.00     23537
weighted avg       1.00      1.00      1.00     23537

M2 - Derived :

Accuracy:  0.9956239112886094
Balanced Accuracy: 0.9948766414643853

Confusion Matrix:
 [[ 9949   103]
 [    0 13485]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99     10052
           1       0.99      1.00      1.00     13485

    accuracy                           1.00     23537
   macro avg       1.00      0.99      1.00     23537
weighted avg       1.00      1.00      1.00     23537

M3 - URL Features :

Accur

### Tuning Hyperparameter:

In [6]:
def SVM_sweep(name, features, C_values=(0.01, 0.1, 1, 10, 100, 1000, 10000)):
    rows = []

    for C in C_values:
        print(f"\n{name} | Linear SVM | C={C}", flush=True)

        svm_pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", LinearSVC(
                C=C,
                max_iter=3000,
                random_state=42
            ))
        ])

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)

            svm_pipeline.fit(X_train[features], y_train)

        converged = not any(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        predictions = svm_pipeline.predict(X_val[features])
        cm = confusion_matrix(y_val, predictions, labels=[0, 1])

        row = {
            "Feature Set": name,
            "C": C,
            "Features": len(features),
            "Accuracy": accuracy_score(y_val, predictions),
            "Balanced Accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "Phishing Recall": recall_score(
                y_val, predictions, pos_label=0
            ),
            "Missed Phishing": int(cm[0, 1]),
            "False Alarms": int(cm[1, 0]),
            "Converged": converged
        }

        rows.append(row)

        print(
            f"Accuracy={row['Accuracy']:.4%} | "
            f"Balanced Accuracy={row['Balanced Accuracy']:.4%} | "
            f"Phishing Recall={row['Phishing Recall']:.4%} | "
            f"Missed Phishing={row['Missed Phishing']} | "
            f"False Alarms={row['False Alarms']} | "
            f"Converged={converged}",
            flush=True
        )

    return pd.DataFrame(rows)

In [9]:
m1_features = feature_sets["M1 - Lexical"]

svm_m1_results = SVM_sweep(
    "M1 - Lexical",
    m1_features
)

svm_m1_results


M1 - Lexical | Linear SVM | C=0.01
Accuracy=88.4225% | Balanced Accuracy=86.6532% | Phishing Recall=74.5225% | Missed Phishing=2561 | False Alarms=164 | Converged=True

M1 - Lexical | Linear SVM | C=0.1
Accuracy=98.9846% | Balanced Accuracy=98.8124% | Phishing Recall=97.6323% | Missed Phishing=238 | False Alarms=1 | Converged=True

M1 - Lexical | Linear SVM | C=1
Accuracy=99.5539% | Balanced Accuracy=99.4777% | Phishing Recall=98.9554% | Missed Phishing=105 | False Alarms=0 | Converged=True

M1 - Lexical | Linear SVM | C=10
Accuracy=99.6856% | Balanced Accuracy=99.6319% | Phishing Recall=99.2638% | Missed Phishing=74 | False Alarms=0 | Converged=True

M1 - Lexical | Linear SVM | C=100
Accuracy=99.7323% | Balanced Accuracy=99.6866% | Phishing Recall=99.3733% | Missed Phishing=63 | False Alarms=0 | Converged=True

M1 - Lexical | Linear SVM | C=1000
Accuracy=99.7281% | Balanced Accuracy=99.6817% | Phishing Recall=99.3633% | Missed Phishing=64 | False Alarms=0 | Converged=True

M1 - Lexic

,Feature Set,C,Features,Accuracy,Balanced Accuracy,Phishing Recall,Missed Phishing,False Alarms,Converged
0,M1 - Lexical,0.01,17,0.884225,0.866532,0.745225,2561,164,True
1,M1 - Lexical,0.10,17,0.989846,0.988124,0.976323,238,1,True
2,M1 - Lexical,1.00,17,0.995539,0.994777,0.989554,105,0,True
3,M1 - Lexical,10.00,17,0.996856,0.996319,0.992638,74,0,True
4,M1 - Lexical,100.00,17,0.997323,0.996866,0.993733,63,0,True
5,M1 - Lexical,1000.00,17,0.997281,0.996817,0.993633,64,0,True
6,M1 - Lexical,10000.00,17,0.997408,0.996966,0.993932,61,0,True


In [10]:
m2_features = feature_sets["M2 - Derived"]

svm_m2_results = SVM_sweep(
    "M2 - Derived",
    m2_features
)

svm_m2_results


M2 - Derived | Linear SVM | C=0.01
Accuracy=88.7836% | Balanced Accuracy=87.1190% | Phishing Recall=75.7063% | Missed Phishing=2442 | False Alarms=198 | Converged=True

M2 - Derived | Linear SVM | C=0.1
Accuracy=98.9718% | Balanced Accuracy=98.8064% | Phishing Recall=97.6721% | Missed Phishing=234 | False Alarms=8 | Converged=True

M2 - Derived | Linear SVM | C=1
Accuracy=99.5624% | Balanced Accuracy=99.4877% | Phishing Recall=98.9753% | Missed Phishing=103 | False Alarms=0 | Converged=True

M2 - Derived | Linear SVM | C=10
Accuracy=99.6983% | Balanced Accuracy=99.6468% | Phishing Recall=99.2937% | Missed Phishing=71 | False Alarms=0 | Converged=True

M2 - Derived | Linear SVM | C=100
Accuracy=99.7408% | Balanced Accuracy=99.6966% | Phishing Recall=99.3932% | Missed Phishing=61 | False Alarms=0 | Converged=True

M2 - Derived | Linear SVM | C=1000
Accuracy=99.7408% | Balanced Accuracy=99.6966% | Phishing Recall=99.3932% | Missed Phishing=61 | False Alarms=0 | Converged=True

M2 - Deriv

,Feature Set,C,Features,Accuracy,Balanced Accuracy,Phishing Recall,Missed Phishing,False Alarms,Converged
0,M2 - Derived,0.01,20,0.887836,0.871190,0.757063,2442,198,True
1,M2 - Derived,0.10,20,0.989718,0.988064,0.976721,234,8,True
2,M2 - Derived,1.00,20,0.995624,0.994877,0.989753,103,0,True
3,M2 - Derived,10.00,20,0.996983,0.996468,0.992937,71,0,True
4,M2 - Derived,100.00,20,0.997408,0.996966,0.993932,61,0,True
5,M2 - Derived,1000.00,20,0.997408,0.996966,0.993932,61,0,True
6,M2 - Derived,10000.00,20,0.997323,0.996866,0.993733,63,0,True


In [11]:
m3_features = feature_sets["M3 - URL Features"]

svm_m3_results = SVM_sweep(
    "M3 - URL Features",
    m3_features
)

svm_m3_results


M3 - URL Features | Linear SVM | C=0.01
Accuracy=99.5539% | Balanced Accuracy=99.4840% | Phishing Recall=99.0052% | Missed Phishing=100 | False Alarms=5 | Converged=True

M3 - URL Features | Linear SVM | C=0.1
Accuracy=99.6176% | Balanced Accuracy=99.5523% | Phishing Recall=99.1047% | Missed Phishing=90 | False Alarms=0 | Converged=True

M3 - URL Features | Linear SVM | C=1
Accuracy=99.6686% | Balanced Accuracy=99.6120% | Phishing Recall=99.2240% | Missed Phishing=78 | False Alarms=0 | Converged=True

M3 - URL Features | Linear SVM | C=10
Accuracy=99.7323% | Balanced Accuracy=99.6866% | Phishing Recall=99.3733% | Missed Phishing=63 | False Alarms=0 | Converged=True

M3 - URL Features | Linear SVM | C=100
Accuracy=99.7366% | Balanced Accuracy=99.6916% | Phishing Recall=99.3832% | Missed Phishing=62 | False Alarms=0 | Converged=True

M3 - URL Features | Linear SVM | C=1000
Accuracy=99.7366% | Balanced Accuracy=99.6916% | Phishing Recall=99.3832% | Missed Phishing=62 | False Alarms=0 | C

,Feature Set,C,Features,Accuracy,Balanced Accuracy,Phishing Recall,Missed Phishing,False Alarms,Converged
0,M3 - URL Features,0.01,21,0.995539,0.994840,0.990052,100,5,True
1,M3 - URL Features,0.10,21,0.996176,0.995523,0.991047,90,0,True
2,M3 - URL Features,1.00,21,0.996686,0.996120,0.992240,78,0,True
3,M3 - URL Features,10.00,21,0.997323,0.996866,0.993733,63,0,True
4,M3 - URL Features,100.00,21,0.997366,0.996916,0.993832,62,0,True
5,M3 - URL Features,1000.00,21,0.997366,0.996916,0.993832,62,0,True
6,M3 - URL Features,10000.00,21,0.997366,0.996916,0.993832,62,0,True


In [12]:
m4_features = feature_sets["M4 - Page-aware"]

svm_m4_results = SVM_sweep(
    "M4 - Page-aware",
    m4_features
)

svm_m4_results


M4 - Page-aware | Linear SVM | C=0.01
Accuracy=99.9405% | Balanced Accuracy=99.9367% | Phishing Recall=99.9105% | Missed Phishing=9 | False Alarms=5 | Converged=True

M4 - Page-aware | Linear SVM | C=0.1
Accuracy=99.9703% | Balanced Accuracy=99.9690% | Phishing Recall=99.9602% | Missed Phishing=4 | False Alarms=3 | Converged=True

M4 - Page-aware | Linear SVM | C=1
Accuracy=99.9915% | Balanced Accuracy=99.9901% | Phishing Recall=99.9801% | Missed Phishing=2 | False Alarms=0 | Converged=True

M4 - Page-aware | Linear SVM | C=10
Accuracy=99.9915% | Balanced Accuracy=99.9913% | Phishing Recall=99.9901% | Missed Phishing=1 | False Alarms=1 | Converged=True

M4 - Page-aware | Linear SVM | C=100
Accuracy=99.9915% | Balanced Accuracy=99.9901% | Phishing Recall=99.9801% | Missed Phishing=2 | False Alarms=0 | Converged=True

M4 - Page-aware | Linear SVM | C=1000
Accuracy=99.9915% | Balanced Accuracy=99.9913% | Phishing Recall=99.9901% | Missed Phishing=1 | False Alarms=1 | Converged=True

M4 -

,Feature Set,C,Features,Accuracy,Balanced Accuracy,Phishing Recall,Missed Phishing,False Alarms,Converged
0,M4 - Page-aware,0.01,49,0.999405,0.999367,0.999105,9,5,True
1,M4 - Page-aware,0.10,49,0.999703,0.999690,0.999602,4,3,True
2,M4 - Page-aware,1.00,49,0.999915,0.999901,0.999801,2,0,True
3,M4 - Page-aware,10.00,49,0.999915,0.999913,0.999901,1,1,True
4,M4 - Page-aware,100.00,49,0.999915,0.999901,0.999801,2,0,True
5,M4 - Page-aware,1000.00,49,0.999915,0.999913,0.999901,1,1,True
6,M4 - Page-aware,10000.00,49,0.999915,0.999913,0.999901,1,1,True


In [13]:
m5_features = feature_sets["M5 - All numeric"]

svm_m5_results = SVM_sweep(
    "M5 - All numeric",
    m5_features
)

svm_m5_results


M5 - All numeric | Linear SVM | C=0.01
Accuracy=100.0000% | Balanced Accuracy=100.0000% | Phishing Recall=100.0000% | Missed Phishing=0 | False Alarms=0 | Converged=True

M5 - All numeric | Linear SVM | C=0.1
Accuracy=100.0000% | Balanced Accuracy=100.0000% | Phishing Recall=100.0000% | Missed Phishing=0 | False Alarms=0 | Converged=True

M5 - All numeric | Linear SVM | C=1
Accuracy=100.0000% | Balanced Accuracy=100.0000% | Phishing Recall=100.0000% | Missed Phishing=0 | False Alarms=0 | Converged=True

M5 - All numeric | Linear SVM | C=10
Accuracy=99.9958% | Balanced Accuracy=99.9950% | Phishing Recall=99.9901% | Missed Phishing=1 | False Alarms=0 | Converged=True

M5 - All numeric | Linear SVM | C=100
Accuracy=99.9958% | Balanced Accuracy=99.9950% | Phishing Recall=99.9901% | Missed Phishing=1 | False Alarms=0 | Converged=True

M5 - All numeric | Linear SVM | C=1000
Accuracy=99.9958% | Balanced Accuracy=99.9950% | Phishing Recall=99.9901% | Missed Phishing=1 | False Alarms=0 | Conve

,Feature Set,C,Features,Accuracy,Balanced Accuracy,Phishing Recall,Missed Phishing,False Alarms,Converged
0,M5 - All numeric,0.01,50,1.000000,1.00000,1.000000,0,0,True
1,M5 - All numeric,0.10,50,1.000000,1.00000,1.000000,0,0,True
2,M5 - All numeric,1.00,50,1.000000,1.00000,1.000000,0,0,True
3,M5 - All numeric,10.00,50,0.999958,0.99995,0.999901,1,0,True
4,M5 - All numeric,100.00,50,0.999958,0.99995,0.999901,1,0,True
5,M5 - All numeric,1000.00,50,0.999958,0.99995,0.999901,1,0,True
6,M5 - All numeric,10000.00,50,0.999958,0.99995,0.999901,1,0,True


### Kernel Trick:

In [ ]:
rbf_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        ))
    ])

for name, features in feature_sets.items():
    start = time.time()
    rbf_pipeline.fit(
        X_train[features],
        y_train
    )
    
    print(f"Training time: {(time.time() - start) / 60:.2f} minutes")
    
    y_val_pred = rbf_pipeline.predict(X_val[features])
    
    cm = confusion_matrix(y_val, y_val_pred, labels=[0, 1])

    print (name, ":\n")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("Balanced accuracy:", balanced_accuracy_score(y_val, y_val_pred))
    print("Phishing recall:", recall_score(y_val, y_val_pred, pos_label=0))
    print("Missed phishing:", cm[0, 1])
    print("False alarms:", cm[1, 0])
    print("Confusion matrix:\n", cm)

In [ ]:
svm = rbf_pipeline.named_steps["svm"]

print("Number of support vectors:", svm.n_support_)
print("Total support vectors:", svm.n_support_.sum())
print("Training samples:", len(X_train))

<hr>